In [ ]:
import os
import sys
import torch
from pathlib import Path
import ultralytics
from ultralytics import YOLO

print("[INFO] Đã nạp xong thư viện cơ bản. Đường dẫn Ultralytics:", ultralytics.__file__)



[INFO] Đã nạp xong thư viện cơ bản. Đường dẫn Ultralytics: c:\Users\ImGey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ultralytics\__init__.py


In [3]:
print("[INFO] Bắt đầu cấy ghép kiến trúc (CoordAtt + MobileNetV3)...")
base_dir = Path(ultralytics.__file__).parent
block_file = base_dir / 'nn' / 'modules' / 'block.py'
init_file  = base_dir / 'nn' / 'modules' / '__init__.py'
tasks_file = base_dir / 'nn' / 'tasks.py'

mobilenet_code = """
import torch
import torch.nn as nn

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.relu = nn.ReLU6(inplace=True)
    def forward(self, x): return self.relu(x + 3) / 6

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.sigmoid = h_sigmoid(inplace=True)
    def forward(self, x): return x * self.sigmoid(x)

class CoordAtt(nn.Module):
    def __init__(self, inp, reduction=32):
        super(CoordAtt, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, inp // reduction)
        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = h_swish()
        self.conv_h = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        out = identity * a_w * a_h
        return out

class BNeck(nn.Module):
    def __init__(self, c1, c2, k, s, hs, se):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(c1, c1, 1, 1, 0, bias=False), nn.BatchNorm2d(c1),
            h_swish() if hs else nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, k, s, k//2, groups=c1, bias=False), nn.BatchNorm2d(c1),
            CoordAtt(c1) if se else nn.Identity(),
            h_swish() if hs else nn.ReLU(inplace=True),
            nn.Conv2d(c1, c2, 1, 1, 0, bias=False), nn.BatchNorm2d(c2)
        )
    def forward(self, x): return self.conv(x)

class Conv_MobileNet(nn.Module):
    def __init__(self, c1, c2, k=3, s=2, p=1):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = h_swish()
    def forward(self, x): return self.act(self.bn(self.conv(x)))
    def forward_fuse(self, x): return self.act(self.conv(x))
"""

with open(block_file, 'r', encoding='utf-8') as f: content = f.read()
if 'CoordAtt' not in content:
    with open(block_file, 'a', encoding='utf-8') as f: f.write("\n" + mobilenet_code)
    
with open(init_file, 'r', encoding='utf-8') as f: content = f.read()
if 'Conv_MobileNet' not in content:
    with open(init_file, 'a', encoding='utf-8') as f: f.write('\nfrom .block import BNeck, Conv_MobileNet\n')

with open(tasks_file, 'r', encoding='utf-8') as f: tasks_content = f.read()
if 'Conv_MobileNet' not in tasks_content:
    tasks_content = "from ultralytics.nn.modules import BNeck, Conv_MobileNet\n" + tasks_content
    tasks_content = tasks_content.replace("Conv,", "Conv, BNeck, Conv_MobileNet,")
    with open(tasks_file, 'w', encoding='utf-8') as f: f.write(tasks_content)

for key in list(sys.modules.keys()):
    if key.startswith('ultralytics'): del sys.modules[key]

print("[OK] Cấy ghép thành công. Sẵn sàng nạp mô hình!")

[INFO] Bắt đầu cấy ghép kiến trúc (CoordAtt + MobileNetV3)...
[OK] Cấy ghép thành công. Sẵn sàng nạp mô hình!


In [4]:
# Chống lỗi bảo mật weights_only khi load file từ máy khác
original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load

# Nạp model
model_path = r'D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.pt'

model = YOLO(model_path)
print(f"[OK] Đã nạp thành công mô hình: {model_path}")

[OK] Đã nạp thành công mô hình: D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.pt


In [5]:
print("--- 1. Đang xuất bản ONNX (FP32 - Độ chính xác gốc) ---")
model.export(format='onnx', simplify=True)
print("[OK] Đã tạo file best.onnx (FP32)")

print("\n--- 2. Đang xuất bản ONNX (FP16 - Lượng tử hóa nhẹ) ---")
model.export(format='onnx', simplify=True, half=True)
print("[OK] Đã ghi đè file best.onnx (FP16)")

--- 1. Đang xuất bản ONNX (FP32 - Độ chính xác gốc) ---
Ultralytics 8.4.68  Python-3.14.2 torch-2.12.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
Ultralytics 8.4.68  Python-3.14.2 torch-2.12.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
YOLOv7-mobilenetv3 summary: 210 layers, 329,853 parameters, 0 gradients, 1.4 GFLOPs
YOLOv7-mobilenetv3 summary: 210 layers, 329,853 parameters, 0 gradients, 1.4 GFLOPs

PyTorch: starting from 'D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (0.9 MB)

PyTorch: starting from 'D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (0.9 MB)

ONNX: starting export with onnx 1.22.0 opset 20...

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with on

In [6]:
print("--- 3. Đang xuất bản NCNN (FP16) ---")
model.export(format='ncnn', half=True)
print("[OK] Đã tạo thư mục best_ncnn_model")

--- 3. Đang xuất bản NCNN (FP16) ---
Ultralytics 8.4.68  Python-3.14.2 torch-2.12.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
Ultralytics 8.4.68  Python-3.14.2 torch-2.12.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
WARNING NCNN export does not support end2end models, disabling end2end branch.
WARNING WARNING NCNN export does not support end2end models, disabling end2end branch.
YOLOv7-mobilenetv3 summary: 210 layers, 329,853 parameters, 0 gradients, 1.4 GFLOPs
YOLOv7-mobilenetv3 summary: 210 layers, 329,853 parameters, 0 gradients, 1.4 GFLOPs

PyTorch: starting from 'D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (0.9 MB)

PyTorch: starting from 'D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (

In [11]:
import os
import time
from ultralytics import YOLO

print("=== BƯỚC 3: TEST VÀ SO SÁNH TỐC ĐỘ CÁC MÔ HÌNH ===")

# 1. HÃY DÁN ĐƯỜNG DẪN THƯ MỤC CHỨA 3 FILE MODEL VÀO ĐÂY:
thu_muc_model = r'D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights'

# 2. HÃY DÁN ĐƯỜNG DẪN THƯ MỤC ẢNH TEST VÀO ĐÂY:
thu_muc_test = r'D:\WorkSpace\DA1\RGBT_Dataset\images\test'

danh_sach_models_goc = [
    'best.pt',              
    'best.onnx',            
    'best_ncnn_model'       
]

# Tự động ghép nối để tạo thành địa chỉ tuyệt đối
danh_sach_models = [os.path.join(thu_muc_model, model) for model in danh_sach_models_goc]

try:
    tat_ca_anh = [f for f in os.listdir(thu_muc_test) if f.lower().endswith(('.jpg', '.png'))]
    anh_test = tat_ca_anh[:50] 
except FileNotFoundError:
    print(f"[LỖI] Không tìm thấy thư mục ảnh: {thu_muc_test}")
    anh_test = []

if len(anh_test) > 0:
    for duong_dan_model in danh_sach_models:
        if not os.path.exists(duong_dan_model):
            print(f"\n[BỎ QUA] Không tìm thấy: '{duong_dan_model}'.")
            continue
            
        print(f"\n---> ĐANG CHẠY THỬ: {os.path.basename(duong_dan_model)} <---")
        model = YOLO(duong_dan_model, task='detect')
        
        print("Đang làm nóng phần cứng...")
        _ = model.predict(source=os.path.join(thu_muc_test, anh_test[0]), imgsz=640, verbose=False)
        
        tong_thoi_gian = 0
        
        for ten_anh in anh_test:
            duong_dan = os.path.join(thu_muc_test, ten_anh)
            
            start_time = time.time()
            _ = model.predict(source=duong_dan, imgsz=640, conf=0.3, verbose=False)
            end_time = time.time()
            
            tong_thoi_gian += (end_time - start_time) * 1000 
            
        thoigian_trungbinh = tong_thoi_gian / len(anh_test)
        fps = 1000 / thoigian_trungbinh
        
        print(f"Hoàn thành test trên {len(anh_test)} ảnh.")
        print(f"Thời gian xử lý trung bình : {thoigian_trungbinh:.2f} ms/ảnh")
        print(f"Tốc độ khung hình (FPS)    : {fps:.2f} FPS")
        print("-" * 40)

=== BƯỚC 3: TEST VÀ SO SÁNH TỐC ĐỘ CÁC MÔ HÌNH ===

---> ĐANG CHẠY THỬ: best.pt <---
Đang làm nóng phần cứng...
Hoàn thành test trên 50 ảnh.
Thời gian xử lý trung bình : 157.60 ms/ảnh
Tốc độ khung hình (FPS)    : 6.35 FPS
----------------------------------------

---> ĐANG CHẠY THỬ: best.onnx <---
Đang làm nóng phần cứng...
Loading D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.onnx for ONNX Runtime inference...
Loading D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights\best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.27.0 with CPUExecutionProvider
Using ONNX Runtime 1.27.0 with CPUExecutionProvider
Hoàn thành test trên 50 ảnh.
Thời gian xử lý trung bình : 87.30 ms/ảnh
Tốc độ khung hình (FPS)    : 11.46 FPS
----------------------------------------

---> ĐANG CHẠY THỬ: best_ncnn_model <---
Đang làm nóng phần cứng...
Loadin

In [15]:
import os
from ultralytics import YOLO

print("=== BƯỚC 4: ĐÁNH GIÁ ĐỘ CHÍNH XÁC (mAP) CÁC MÔ HÌNH ===")

# 1. ĐƯỜNG DẪN THƯ MỤC MODEL VÀ YAML
thu_muc_model = r'D:\WorkSpace\DA1\Drone_Project\Bao_Cao_YOLOv7_Hybrid_BW\kaggle\working\Drone_YOLOv7_Hybrid_BW\train_v7_mobilenet_bw\weights'
duong_dan_yaml = r'D:\WorkSpace\DA1\Drone_Project\data_drone_bw.yaml'

# 2. Danh sách model
danh_sach_models_goc = ['best.pt', 'best.onnx']
danh_sach_models = [os.path.join(thu_muc_model, model) for model in danh_sach_models_goc]

for duong_dan_model in danh_sach_models:
    if not os.path.exists(duong_dan_model):
        print(f"\n[BỎ QUA] Không tìm thấy: '{duong_dan_model}'.")
        continue
        
    try:
        ten_model = os.path.basename(duong_dan_model)
        print(f"\n---> ĐANG CHẤM ĐIỂM MÔ HÌNH: {ten_model} <---")
        
        # Tắt verbose để ẩn các log rác của YOLO, chỉ giữ lại bảng điểm cuối cùng
        model = YOLO(duong_dan_model, task='detect')
        metrics = model.val(data=duong_dan_yaml, split='val', verbose=False)
        
        # Bóc tách dữ liệu
        class_names = metrics.names
        p_per_class = metrics.box.p
        r_per_class = metrics.box.r
        map50_per_class = metrics.box.ap50
        map50_95_per_class = metrics.box.ap

        # --- PHẦN IN KẾT QUẢ ĐÃ ĐƯỢC ÉP HIỂN THỊ FULL ---
        print(f"\n{'='*65}")
        print(f"BẢNG ĐÁNH GIÁ ĐỘ CHÍNH XÁC ({ten_model}):")
        print(f"{'Class':<12} | {'Precision':<10} | {'Recall':<10} | {'mAP@50':<10} | {'mAP@50-95':<10}")
        print("-" * 65)
        
        # In dòng ALL
        print(f"{'all':<12} | {metrics.box.mp:<10.4f} | {metrics.box.mr:<10.4f} | {metrics.box.map50:<10.4f} | {metrics.box.map:<10.4f}")
        
        # In từng Class một
        for i, c_id in enumerate(metrics.box.ap_class_index):
            ten_class = class_names[c_id]
            precision = p_per_class[i]
            recall = r_per_class[i]
            map50 = map50_per_class[i]
            map50_95 = map50_95_per_class[i]
            print(f"{ten_class:<12} | {precision:<10.4f} | {recall:<10.4f} | {map50:<10.4f} | {map50_95:<10.4f}")
            
        print(f"{'='*65}\n")
        
    except Exception as e:
        print(f"[LỖI] Không thể đánh giá {ten_model}. Lỗi: {e}")

=== BƯỚC 4: ĐÁNH GIÁ ĐỘ CHÍNH XÁC (mAP) CÁC MÔ HÌNH ===

---> ĐANG CHẤM ĐIỂM MÔ HÌNH: best.pt <---
Ultralytics 8.4.68  Python-3.14.2 torch-2.12.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
Ultralytics 8.4.68  Python-3.14.2 torch-2.12.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
YOLOv7-mobilenetv3 summary: 226 layers, 332,253 parameters, 0 gradients, 1.4 GFLOPs
YOLOv7-mobilenetv3 summary: 226 layers, 332,253 parameters, 0 gradients, 1.4 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 64.529.0 MB/s, size: 62.5 KB)
val: Fast image access  (ping: 0.10.1 ms, read: 64.529.0 MB/s, size: 62.5 KB)
val: Scanning D:\WorkSpace\DA1\RGBT_Dataset\labels\test.cache... 3366 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3366/3366 1.0Git/s 0.0s
val: D:\WorkSpace\DA1\RGBT_Dataset\images\test\video2_frame_00294.jpg: 1 duplicate labels removed
val: D:\WorkSpace\DA1\RGBT_Dataset\images\test\video2_frame_00294.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P        